In [4]:
from langgraph.graph import StateGraph, END
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage
from typing import TypedDict
import re
import os
from langchain_ollama import ChatOllama


# ===== LLM =====
llm = ChatOllama(
    model="llama3.2",
    temperature=0.7
)

print("Setup done.")

Setup done.


In [5]:
# ===== STATE =====
class ResearchState(TypedDict):
    question: str
    answer: str
    iterations: int
    passed: bool

# ===== NODE 1: GENERATE =====
def generate(state: ResearchState) -> ResearchState:
    print(f"\n--- GENERATE NODE (iteration {state['iterations'] + 1}) ---")
    question = state["question"]
    
    response = llm.invoke([
        HumanMessage(content=f"""Answer this research question in detail. 
        Provide a comprehensive answer with specific facts and examples.
        
        Question: {question}
        
        Answer:""")
    ])
    
    answer = response.content
    print(f"Generated answer ({len(answer.split())} words)")
    
    return {
        "question": question,
        "answer": answer,
        "iterations": state["iterations"],
        "passed": False
    }

# ===== NODE 2: EVALUATE =====
def evaluate(state: ResearchState) -> ResearchState:
    print(f"\n--- EVALUATE NODE ---")
    answer = state["answer"]
    word_count = len(answer.split())
    
    passed = word_count > 1000
    
    print(f"Word count: {word_count}")
    print(f"Evaluation: {'PASSED ✅' if passed else 'FAILED ❌ — needs refinement'}")
    
    return {
        **state,
        "passed": passed
    }

# ===== NODE 3: REFINE =====
def refine(state: ResearchState) -> ResearchState:
    print(f"\n--- REFINE NODE ---")
    question = state["question"]
    current_answer = state["answer"]
    
    response = llm.invoke([
        HumanMessage(content=f"""Your previous answer was too short and needs improvement.
        Expand it significantly with more details, examples, and explanations.
        
        Question: {question}
        
        Previous answer: {current_answer}
        
        Improved answer:""")
    ])
    
    refined_answer = response.content
    print(f"Refined answer ({len(refined_answer.split())} words)")
    
    return {
        **state,
        "answer": refined_answer,
        "iterations": state["iterations"] + 1
    }

# ===== CONDITIONAL EDGE =====
def condition(state: ResearchState) -> str:
    if state["passed"]:
        return "end"
    elif state["iterations"] >= 3: 
        print("Max iterations reached, stopping.")
        return "end"
    else:
        return "refine"

# ===== BUILD GRAPH =====
graph = StateGraph(ResearchState)

graph.add_node("generate", generate)
graph.add_node("evaluate", evaluate)
graph.add_node("refine", refine)

graph.set_entry_point("generate")
graph.add_edge("generate", "evaluate")
graph.add_conditional_edges(
    "evaluate",
    condition,
    {
        "end": END,
        "refine": "refine"
    }
)
graph.add_edge("refine", "evaluate")

app = graph.compile()
print("Graph compiled successfully.")

Graph compiled successfully.


In [6]:
#Running the graph 
initial_state = {
    "question": "What are the main causes and effects of climate change?",
    "answer": "",
    "iterations": 0,
    "passed": False
}

print("Starting LangGraph workflow...")
print(f"Question: {initial_state['question']}\n")

final_state = app.invoke(initial_state)

print("\n" + "="*60)
print("FINAL STATE:")
print("="*60)
print(f"Question: {final_state['question']}")
print(f"Iterations: {final_state['iterations']}")
print(f"Passed: {final_state['passed']}")
print(f"\nFinal Answer:\n{final_state['answer']}")

Starting LangGraph workflow...
Question: What are the main causes and effects of climate change?


--- GENERATE NODE (iteration 1) ---
Generated answer (607 words)

--- EVALUATE NODE ---
Word count: 607
Evaluation: FAILED ❌ — needs refinement

--- REFINE NODE ---
Refined answer (1175 words)

--- EVALUATE NODE ---
Word count: 1175
Evaluation: PASSED ✅

FINAL STATE:
Question: What are the main causes and effects of climate change?
Iterations: 1
Passed: True

Final Answer:
Climate change is one of the most pressing issues of our time, with far-reaching consequences for the environment, human health, and the economy. The main causes and effects of climate change can be understood by examining the scientific evidence and exploring the complex interactions between natural and anthropogenic factors.

**Causes of Climate Change:**

1. **Greenhouse gases:** Human activities, such as burning fossil fuels (coal, oil, and gas), deforestation, and land-use changes, have increased the concentration 